# 03 — Implicit ALS with Hybrid Interaction Strength

Trains an implicit ALS model on Goodreads hybrid interaction signals:
1. Load train/val/test splits from GCS (schema: user_id, book_id, is_read, rating)
2. Compute interaction strength: `r = 1 + 2*is_read + β*max(0, rating - 3)`
3. Grid search over ALS hyperparameters (rank, regParam, maxIter, β)
4. Use `implicitPrefs=True` — r becomes the confidence weight
5. Evaluate with ranking metrics: Precision@K, Recall@K, NDCG@K, MAP@K
6. Save best model and results to GCS

With `implicitPrefs=True`, PySpark ALS interprets the "rating" column as confidence:
- Preference p_ui = 1 if r_ui > 0 (always true since min r = 1)
- Confidence c_ui = 1 + α * r_ui (α defaults to 1.0)

In [15]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = os.path.expanduser(
    "~/.config/gcloud/application_default_credentials.json"
)

import math
import time
from itertools import product

from pyspark.sql import SparkSession, Window
import pyspark.sql.functions as F
from pyspark.ml.recommendation import ALS

spark = (
    SparkSession.builder
    .appName("goodreads_als")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY")
    .config("spark.hadoop.fs.gs.impl",
            "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem")
    .config("spark.hadoop.fs.gs.auth.type", "APPLICATION_DEFAULT")
    .getOrCreate()
)

GCS_BASE = "gs://nshen7-personal-bucket/projects/rec_sys_goodreads"

# --- Toggle: set to False for full dataset ---
SAMPLE_MODE = True

SPLITS_BASE = f"{GCS_BASE}/data/splits_sample_mode" if SAMPLE_MODE else f"{GCS_BASE}/data/splits"
MODEL_BASE = f"{GCS_BASE}/models/als_sample_mode" if SAMPLE_MODE else f"{GCS_BASE}/models/als"

print(f"SAMPLE_MODE: {SAMPLE_MODE}")
print(f"SPLITS_BASE: {SPLITS_BASE}")
print(f"MODEL_BASE:  {MODEL_BASE}")

SAMPLE_MODE: True
SPLITS_BASE: gs://nshen7-personal-bucket/projects/rec_sys_goodreads/data/splits_sample_mode
MODEL_BASE:  gs://nshen7-personal-bucket/projects/rec_sys_goodreads/models/als_sample_mode


## 1. Load Splits

In [2]:
train = spark.read.parquet(f"{SPLITS_BASE}/train").cache()
val = spark.read.parquet(f"{SPLITS_BASE}/val").cache()
test = spark.read.parquet(f"{SPLITS_BASE}/test")

n_train = train.count()
n_val = val.count()
n_test = test.count()

print(f"Train: {n_train:,} interactions")
print(f"Val:   {n_val:,} interactions")
print(f"Test:  {n_test:,} interactions")
print()
train.printSchema()
train.show(5)

assert set(train.columns) == {"user_id", "book_id", "is_read", "rating"}, \
    f"Unexpected columns: {train.columns}"

Train: 37,224 interactions
Val:   6,032 interactions
Test:  2,062 interactions

root
 |-- user_id: integer (nullable = true)
 |-- book_id: integer (nullable = true)
 |-- is_read: integer (nullable = true)
 |-- rating: integer (nullable = true)

+-------+-------+-------+------+
|user_id|book_id|is_read|rating|
+-------+-------+-------+------+
| 272319|    790|      0|     0|
| 380519|  19552|      0|     0|
| 140656| 128216|      0|     0|
| 215027|  69747|      0|     0|
|   7644|  58533|      1|     5|
+-------+-------+-------+------+
only showing top 5 rows


## 2. Interaction Strength & Popularity Baseline

The hybrid interaction strength signal:

    r = 1·[shelved] + 2·[read] + β · max(0, rating - 3)

- Every row is a shelf event → minimum r = 1
- `is_read` adds 2 to the confidence
- Ratings above 3 add β · (rating - 3)
- β is a tunable hyperparameter grid-searched alongside ALS params

In [4]:
def compute_r(df, beta):
    """Compute interaction strength r = 1 + 2*is_read + beta*max(0, rating - 3)."""
    return df.withColumn(
        "r",
        F.lit(1)
        + F.lit(2) * F.col("is_read")
        + F.lit(beta) * F.greatest(F.lit(0), F.col("rating") - F.lit(3)),
    )

# Show r distribution for a few beta values
for beta in [0.0, 0.5, 1.0, 2.0]:
    train_r = compute_r(train, beta)
    stats = train_r.select(
        F.mean("r").alias("mean_r"),
        F.stddev("r").alias("std_r"),
        F.min("r").alias("min_r"),
        F.max("r").alias("max_r"),
    ).collect()[0]
    print(f"beta={beta}: mean={stats['mean_r']:.3f}, std={stats['std_r']:.3f}, "
          f"min={stats['min_r']:.1f}, max={stats['max_r']:.1f}")
    train_r.groupBy("r").count().orderBy("r").show(truncate=False)

beta=0.0: mean=1.910, std=0.996, min=1.0, max=3.0
+---+-----+
|r  |count|
+---+-----+
|1.0|20287|
|3.0|16937|
+---+-----+

beta=0.5: mean=2.131, std=1.267, min=1.0, max=4.0
+---+-----+
|r  |count|
+---+-----+
|1.0|20287|
|3.0|5784 |
|3.5|5883 |
|4.0|5270 |
+---+-----+

beta=1.0: mean=2.351, std=1.576, min=1.0, max=5.0
+---+-----+
|r  |count|
+---+-----+
|1.0|20287|
|3.0|5784 |
|4.0|5883 |
|5.0|5270 |
+---+-----+

beta=2.0: mean=2.792, std=2.244, min=1.0, max=7.0
+---+-----+
|r  |count|
+---+-----+
|1.0|20287|
|3.0|5784 |
|5.0|5883 |
|7.0|5270 |
+---+-----+



In [5]:
# --- Popularity baseline ---
def compute_popularity_baseline(train, val, K):
    popular_books = (
        train.groupBy("book_id")
        .agg(F.count("*").alias("n_interactions"))
        .orderBy(F.desc("n_interactions"))
        .limit(K)
        .select("book_id")
        .collect()
    )
    popular_book_ids = set(row["book_id"] for row in popular_books)

    ground_truth_val = val.select("user_id", "book_id")
    n_val_users = val.select("user_id").distinct().count()
    pop_hits_total = ground_truth_val.filter(F.col("book_id").isin(popular_book_ids)).count()
    pop_precision = pop_hits_total / (n_val_users * K)
    print(f"\nPopularity baseline Precision@{K}: {pop_precision:.4f}")

for K in [10, 20, 50]:
    compute_popularity_baseline(train, val, K)


Popularity baseline Precision@10: 0.0005

Popularity baseline Precision@20: 0.0006

Popularity baseline Precision@50: 0.0007


## 3. Implicit ALS Grid Search on Validation Set

Grid: rank × regParam × maxIter × β

Primary metric: NDCG@K. Also computes Precision@K, Recall@K, MAP@K.

In [10]:
def evaluate_ranking(model, eval_df, beta, k=10):
    """
    Compute Precision@K, Recall@K, NDCG@K (graded), MAP@K for implicit feedback.

    Ground truth: all (user, book) pairs in eval_df.
    NDCG uses graded relevance: r = 1 + 2*is_read + beta*max(0, rating-3).
    """
    eval_users = eval_df.select("user_id").distinct()

    # Compute graded relevance on eval set (model never saw this data)
    eval_with_r = compute_r(eval_df, beta)
    ground_truth_r = eval_with_r.select("user_id", "book_id", "r")

    # Generate top-K recommendations
    recs = model.recommendForUserSubset(eval_users, k)
    recs_exploded = (
        recs
        .select("user_id", F.posexplode("recommendations").alias("rec_rank", "rec"))
        .select("user_id", "rec_rank", F.col("rec.book_id").alias("rec_book_id"))
    )

    # Hits: recommended items that appear in ground truth (with relevance score)
    hits = recs_exploded.join(
        ground_truth_r,
        (recs_exploded["user_id"] == ground_truth_r["user_id"])
        & (recs_exploded["rec_book_id"] == ground_truth_r["book_id"]),
        "inner",
    ).select(recs_exploded["user_id"], "rec_book_id", "rec_rank", "r")

    hits_per_user = hits.groupBy("user_id").agg(F.count("*").alias("n_hits"))

    n_relevant_per_user = ground_truth_r.groupBy("user_id").agg(
        F.count("*").alias("n_relevant")
    )

    # --- Precision@K ---
    precision_per_user = (
        eval_users
        .join(hits_per_user, "user_id", "left")
        .fillna(0, subset=["n_hits"])
        .withColumn("precision_at_k", F.col("n_hits") / F.lit(k))
    )
    avg_precision_at_k = precision_per_user.select(F.mean("precision_at_k")).collect()[0][0]

    # --- Recall@K ---
    recall_per_user = (
        eval_users
        .join(hits_per_user, "user_id", "left")
        .fillna(0, subset=["n_hits"])
        .join(n_relevant_per_user, "user_id", "left")
        .fillna(1, subset=["n_relevant"])
        .withColumn("recall_at_k", F.col("n_hits") / F.col("n_relevant"))
    )
    avg_recall_at_k = recall_per_user.select(F.mean("recall_at_k")).collect()[0][0]

    # --- NDCG@K (graded relevance) ---
    # DCG: sum r_i / log2(rank + 2) for each hit
    dcg_per_user = (
        hits.select("user_id", "rec_rank", "r")
        .withColumn("dcg_i", F.col("r") / F.log2(F.col("rec_rank") + 2))
        .groupBy("user_id")
        .agg(F.sum("dcg_i").alias("dcg"))
    )

    # IDCG: sort each user's eval items by r descending, take top-K, sum r_i / log2(i+2)
    w_ideal = Window.partitionBy("user_id").orderBy(F.desc("r"))
    ideal_ranking = (
        ground_truth_r
        .withColumn("ideal_rank", F.row_number().over(w_ideal) - 1)  # 0-indexed
        .filter(F.col("ideal_rank") < k)
        .withColumn("idcg_i", F.col("r") / F.log2(F.col("ideal_rank") + 2))
        .groupBy("user_id")
        .agg(F.sum("idcg_i").alias("idcg"))
    )

    ndcg_per_user = (
        eval_users
        .join(dcg_per_user, "user_id", "left")
        .fillna(0.0, subset=["dcg"])
        .join(ideal_ranking, "user_id", "left")
        .fillna(1.0, subset=["idcg"])
        .withColumn("ndcg", F.col("dcg") / F.col("idcg"))
    )
    avg_ndcg = ndcg_per_user.select(F.mean("ndcg")).collect()[0][0]

    # --- MAP@K ---
    # Build a separate ground truth set with just (user_id, book_id) aliased to avoid ambiguity
    gt_for_map = ground_truth_r.select(
        F.col("user_id").alias("gt_user_id"),
        F.col("book_id").alias("gt_book_id"),
    )

    all_ranks = recs_exploded.select("user_id", "rec_rank", "rec_book_id")
    hits_marked = (
        all_ranks
        .join(
            gt_for_map,
            (all_ranks["user_id"] == gt_for_map["gt_user_id"])
            & (all_ranks["rec_book_id"] == gt_for_map["gt_book_id"]),
            "left",
        )
        .select(
            all_ranks["user_id"],
            "rec_rank",
            F.when(F.col("gt_book_id").isNotNull(), F.lit(1)).otherwise(F.lit(0)).alias("is_hit"),
        )
    )

    w = Window.partitionBy("user_id").orderBy("rec_rank").rowsBetween(
        Window.unboundedPreceding, Window.currentRow
    )
    hits_cumul = (
        hits_marked
        .withColumn("cum_hits", F.sum("is_hit").over(w))
        .withColumn("prec_at_i", F.col("cum_hits") / (F.col("rec_rank") + 1))
        .withColumn("ap_contrib", F.col("prec_at_i") * F.col("is_hit"))
    )

    ap_per_user = hits_cumul.groupBy("user_id").agg(F.sum("ap_contrib").alias("sum_ap"))

    n_rel_for_map = ground_truth_r.groupBy("user_id").agg(
        F.least(F.count("*"), F.lit(k)).alias("n_rel")
    )
    map_per_user = (
        eval_users
        .join(ap_per_user, "user_id", "left")
        .fillna(0.0, subset=["sum_ap"])
        .join(n_rel_for_map, "user_id", "left")
        .fillna(1, subset=["n_rel"])
        .withColumn("ap", F.col("sum_ap") / F.col("n_rel"))
    )
    avg_map = map_per_user.select(F.mean("ap")).collect()[0][0]

    return {
        f"precision@{k}": avg_precision_at_k,
        f"recall@{k}": avg_recall_at_k,
        f"ndcg@{k}": avg_ndcg,
        f"map@{k}": avg_map,
    }


# --- Grid search ---

K = 10

param_grid = {
    "rank": [10, 50],
    "regParam": [0.01, 0.1, 1.0],
    "maxIter": [15],
    "beta": [0.0, 0.5, 1.0],
}

results = []
combos = list(product(
    param_grid["rank"],
    param_grid["regParam"],
    param_grid["maxIter"],
    param_grid["beta"],
))

for i, (rank, reg, max_iter, beta) in enumerate(combos, 1):
    print(f"[{i}/{len(combos)}] rank={rank}, regParam={reg}, "
          f"maxIter={max_iter}, beta={beta} ... ", end="", flush=True)
    t0 = time.time()

    train_r = compute_r(train, beta).select(
        "user_id", "book_id", F.col("r").cast("float").alias("rating")
    )

    als = ALS(
        rank=rank,
        maxIter=max_iter,
        regParam=reg,
        userCol="user_id",
        itemCol="book_id",
        ratingCol="rating",
        implicitPrefs=True,
        coldStartStrategy="drop",
        seed=42,
    )

    model = als.fit(train_r)
    metrics = evaluate_ranking(model, val, beta, k=K)

    elapsed = time.time() - t0
    result = {
        "rank": rank,
        "regParam": reg,
        "maxIter": max_iter,
        "beta": beta,
        **metrics,
        "elapsed_s": round(elapsed, 1),
    }
    results.append(result)
    print(f"NDCG@{K}={metrics[f'ndcg@{K}']:.4f}, "
          f"P@{K}={metrics[f'precision@{K}']:.4f}, "
          f"R@{K}={metrics[f'recall@{K}']:.4f}, "
          f"MAP@{K}={metrics[f'map@{K}']:.4f} ({elapsed:.0f}s)")

print("\nGrid search complete.")

[1/18] rank=10, regParam=0.01, maxIter=15, beta=0.0 ... 

NDCG@10=0.0017, P@10=0.0006, R@10=0.0027, MAP@10=0.0009 (18s)
[2/18] rank=10, regParam=0.01, maxIter=15, beta=0.5 ... 

NDCG@10=0.0021, P@10=0.0007, R@10=0.0035, MAP@10=0.0010 (18s)
[3/18] rank=10, regParam=0.01, maxIter=15, beta=1.0 ... 

NDCG@10=0.0025, P@10=0.0009, R@10=0.0042, MAP@10=0.0012 (19s)
[4/18] rank=10, regParam=0.1, maxIter=15, beta=0.0 ... 

NDCG@10=0.0017, P@10=0.0006, R@10=0.0026, MAP@10=0.0009 (20s)
[5/18] rank=10, regParam=0.1, maxIter=15, beta=0.5 ... 

...

NDCG@10=0.0012, P@10=0.0005, R@10=0.0026, MAP@10=0.0005 (21s)
[15/18] rank=50, regParam=0.1, maxIter=15, beta=1.0 ... 

NDCG@10=0.0014, P@10=0.0006, R@10=0.0031, MAP@10=0.0007 (23s)
[16/18] rank=50, regParam=1.0, maxIter=15, beta=0.0 ... 

NDCG@10=0.0009, P@10=0.0003, R@10=0.0019, MAP@10=0.0005 (22s)
[17/18] rank=50, regParam=1.0, maxIter=15, beta=0.5 ... 

NDCG@10=0.0014, P@10=0.0005, R@10=0.0027, MAP@10=0.0007 (21s)
[18/18] rank=50, regParam=1.0, maxIter=15, beta=1.0 ... 

NDCG@10=0.0014, P@10=0.0005, R@10=0.0027, MAP@10=0.0008 (22s)

Grid search complete.


## 4. Best Model Summary

In [11]:
# Sort by NDCG@K (primary metric) descending
results_sorted = sorted(results, key=lambda x: -x[f"ndcg@{K}"])

print(f"{'Rank':>6} {'RegParam':>10} {'MaxIter':>8} {'Beta':>6} "
      f"{'NDCG@10':>10} {'P@10':>8} {'R@10':>8} {'MAP@10':>10}")
print("-" * 72)
for r in results_sorted[:20]:  # show top 20
    print(f"{r['rank']:>6} {r['regParam']:>10.3f} {r['maxIter']:>8} {r['beta']:>6.1f} "
          f"{r[f'ndcg@{K}']:>10.4f} {r[f'precision@{K}']:>8.4f} "
          f"{r[f'recall@{K}']:>8.4f} {r[f'map@{K}']:>10.4f}")

best = results_sorted[0]
print(f"\nBest: rank={best['rank']}, regParam={best['regParam']}, "
      f"maxIter={best['maxIter']}, beta={best['beta']}")
print(f"Best Val NDCG@{K}: {best[f'ndcg@{K}']:.4f}")

  Rank   RegParam  MaxIter   Beta    NDCG@10     P@10     R@10     MAP@10
------------------------------------------------------------------------
    10      1.000       15    1.0     0.0028   0.0008   0.0041     0.0017
    10      0.100       15    1.0     0.0026   0.0009   0.0044     0.0013
    10      0.010       15    1.0     0.0025   0.0009   0.0042     0.0012
    10      1.000       15    0.5     0.0023   0.0007   0.0031     0.0013
    10      0.100       15    0.5     0.0022   0.0008   0.0041     0.0011
    10      0.010       15    0.5     0.0021   0.0007   0.0035     0.0010
    10      0.010       15    0.0     0.0017   0.0006   0.0027     0.0009
    10      0.100       15    0.0     0.0017   0.0006   0.0026     0.0009
    10      1.000       15    0.0     0.0016   0.0005   0.0023     0.0008
    50      0.010       15    1.0     0.0015   0.0006   0.0035     0.0007
    50      1.000       15    1.0     0.0014   0.0005   0.0027     0.0008
    50      1.000       15    0.5     0

## 5. Evaluate Best Model on Test Set (Ranking Metrics)

In [12]:
# Retrain with best hyperparameters
best_beta = best["beta"]
train_r = compute_r(train, best_beta).select(
    "user_id", "book_id", F.col("r").cast("float").alias("rating")
)

best_als = ALS(
    rank=best["rank"],
    maxIter=best["maxIter"],
    regParam=best["regParam"],
    userCol="user_id",
    itemCol="book_id",
    ratingCol="rating",
    implicitPrefs=True,
    coldStartStrategy="drop",
    seed=42,
)
best_model = best_als.fit(train_r)

# Evaluate on test set
test_metrics = evaluate_ranking(best_model, test, best_beta, k=K)

print(f"Test Ranking Metrics (K={K}):")
for metric_name, metric_val in test_metrics.items():
    print(f"  {metric_name}: {metric_val:.4f}")

Test Ranking Metrics (K=10):
  precision@10: 0.0005
  recall@10: 0.0029
  ndcg@10: 0.0011
  map@10: 0.0006


## 6. Metrics at Multiple K Values

In [14]:
# Evaluate at multiple K values
for k_val in [10, 50]:
    metrics_k = evaluate_ranking(best_model, test, best_beta, k=k_val)
    print(f"K={k_val:>2}: " + ", ".join(f"{k}={v:.4f}" for k, v in metrics_k.items()))

K=10: precision@10=0.0005, recall@10=0.0029, ndcg@10=0.0011, map@10=0.0006


K=50: precision@50=0.0004, recall@50=0.0131, ndcg@50=0.0033, map@50=0.0010


## 7. Save Model & Results

In [16]:
# Save ALS model
best_model.write().overwrite().save(f"{MODEL_BASE}/model")
print(f"Model saved to: {MODEL_BASE}/model")

# Save tuning results
tuning_df = spark.createDataFrame(results_sorted)
tuning_df.coalesce(1).write.mode("overwrite").parquet(f"{MODEL_BASE}/tuning_results")
print(f"Tuning results saved to: {MODEL_BASE}/tuning_results")

Model saved to: gs://nshen7-personal-bucket/projects/rec_sys_goodreads/models/als_sample_mode/model


Tuning results saved to: gs://nshen7-personal-bucket/projects/rec_sys_goodreads/models/als_sample_mode/tuning_results


## 8. Summary

In [17]:
print("=" * 60)
print("  IMPLICIT ALS MODEL SUMMARY")
print("=" * 60)
print(f"Sample mode:         {SAMPLE_MODE}")
print()
print(f"Data:")
print(f"  Train:             {n_train:,} interactions")
print(f"  Val:               {n_val:,} interactions")
print(f"  Test:              {n_test:,} interactions")
print()
print(f"Best hyperparameters:")
print(f"  rank:              {best['rank']}")
print(f"  regParam:          {best['regParam']}")
print(f"  maxIter:           {best['maxIter']}")
print(f"  beta:              {best['beta']}")
print()
print(f"Interaction strength: r = 1 + 2*is_read + {best['beta']}*max(0, rating-3)")
print()
print(f"Validation metrics (K={K}):")
print(f"  NDCG@{K}:           {best[f'ndcg@{K}']:.4f}")
print(f"  Precision@{K}:      {best[f'precision@{K}']:.4f}")
print(f"  Recall@{K}:         {best[f'recall@{K}']:.4f}")
print(f"  MAP@{K}:            {best[f'map@{K}']:.4f}")
print()
print(f"Test metrics (K={K}):")
for metric_name, metric_val in test_metrics.items():
    print(f"  {metric_name}:{'':>{14-len(metric_name)}}{metric_val:.4f}")
print()
print(f"Output paths:")
print(f"  Model:             {MODEL_BASE}/model")
print(f"  Tuning results:    {MODEL_BASE}/tuning_results")

  IMPLICIT ALS MODEL SUMMARY
Sample mode:         True

Data:
  Train:             37,224 interactions
  Val:               6,032 interactions
  Test:              2,062 interactions

Best hyperparameters:
  rank:              10
  regParam:          1.0
  maxIter:           15
  beta:              1.0

Interaction strength: r = 1 + 2*is_read + 1.0*max(0, rating-3)

Validation metrics (K=10):
  NDCG@10:           0.0028
  Precision@10:      0.0008
  Recall@10:         0.0041
  MAP@10:            0.0017

Test metrics (K=10):
  precision@10:  0.0005
  recall@10:     0.0029
  ndcg@10:       0.0011
  map@10:        0.0006

Output paths:
  Model:             gs://nshen7-personal-bucket/projects/rec_sys_goodreads/models/als_sample_mode/model
  Tuning results:    gs://nshen7-personal-bucket/projects/rec_sys_goodreads/models/als_sample_mode/tuning_results


## 9. Example Recommendations

In [18]:
# Pick 3 sample users from the test set
sample_users = test.select("user_id").distinct().limit(3)
sample_user_ids = [row["user_id"] for row in sample_users.collect()]

for uid in sample_user_ids:
    print(f"\n{'=' * 50}")
    print(f"User {uid}")
    print(f"{'=' * 50}")

    # Training history
    user_train = train.filter(F.col("user_id") == uid)
    user_train_r = compute_r(user_train, best_beta)
    print(f"\nTraining interactions ({user_train.count()}):")
    user_train_r.select("book_id", "is_read", "rating", "r").show(10, truncate=False)

    # Test set (actual)
    user_test = test.filter(F.col("user_id") == uid)
    print(f"Test interactions ({user_test.count()}):")
    user_test.show(10, truncate=False)

    # Top-10 recommendations
    user_df = spark.createDataFrame([(uid,)], ["user_id"])
    user_recs = best_model.recommendForUserSubset(user_df, 10)
    print("Top-10 recommendations:")
    (
        user_recs
        .select(F.posexplode("recommendations").alias("rank", "rec"))
        .select(
            (F.col("rank") + 1).alias("rank"),
            F.col("rec.book_id").alias("book_id"),
            F.round(F.col("rec.rating"), 4).alias("predicted_score"),
        )
        .show(10, truncate=False)
    )


User 83250

Training interactions (1):
+-------+-------+------+---+
|book_id|is_read|rating|r  |
+-------+-------+------+---+
|8000   |0      |0     |1.0|
+-------+-------+------+---+



Test interactions (3):


+-------+-------+-------+------+
|user_id|book_id|is_read|rating|
+-------+-------+-------+------+
|83250  |16925  |0      |0     |
|83250  |703    |0      |0     |
|83250  |48034  |0      |0     |
+-------+-------+-------+------+

Top-10 recommendations:


+----+-------+---------------+
|rank|book_id|predicted_score|
+----+-------+---------------+
|1   |8000   |0.0561         |
|2   |786    |0.0543         |
|3   |1605   |0.0472         |
|4   |16247  |0.0451         |
|5   |438    |0.0412         |
|6   |997    |0.0338         |
|7   |1000   |0.0315         |
|8   |670    |0.031          |
|9   |5467   |0.0295         |
|10  |7348   |0.0295         |
+----+-------+---------------+


User 419543

Training interactions (7):
+-------+-------+------+---+
|book_id|is_read|rating|r  |
+-------+-------+------+---+
|21532  |0      |0     |1.0|
|4761   |1      |5     |5.0|
|3295   |0      |0     |1.0|
|6971   |1      |5     |5.0|
|6915   |0      |0     |1.0|
|4902   |0      |0     |1.0|
|2867   |0      |0     |1.0|
+-------+-------+------+---+



Test interactions (2):


+-------+-------+-------+------+
|user_id|book_id|is_read|rating|
+-------+-------+-------+------+
|419543 |36505  |0      |0     |
|419543 |36751  |0      |0     |
+-------+-------+-------+------+

Top-10 recommendations:
+----+-------+---------------+
|rank|book_id|predicted_score|
+----+-------+---------------+
|1   |1387   |0.128          |
|2   |66     |0.122          |
|3   |938    |0.1167         |
|4   |6989   |0.107          |
|5   |1604   |0.0832         |
|6   |7008   |0.0824         |
|7   |461    |0.0749         |
|8   |6971   |0.0737         |
|9   |1574   |0.0716         |
|10  |996    |0.0643         |
+----+-------+---------------+


User 390467

Training interactions (3):
+-------+-------+------+---+
|book_id|is_read|rating|r  |
+-------+-------+------+---+
|1614   |0      |0     |1.0|
|33902  |0      |0     |1.0|
|1000   |1      |5     |5.0|
+-------+-------+------+---+



Test interactions (1):


+-------+-------+-------+------+
|user_id|book_id|is_read|rating|
+-------+-------+-------+------+
|390467 |17631  |0      |0     |
+-------+-------+-------+------+

Top-10 recommendations:
+----+-------+---------------+
|rank|book_id|predicted_score|
+----+-------+---------------+
|1   |1000   |0.323          |
|2   |1117   |0.1747         |
|3   |997    |0.1531         |
|4   |536    |0.1422         |
|5   |1611   |0.1402         |
|6   |13321  |0.1267         |
|7   |5465   |0.1182         |
|8   |938    |0.1072         |
|9   |6410   |0.1067         |
|10  |439    |0.101          |
+----+-------+---------------+

